## Лабораторная работа №3

Цель: реализовать многослойную нейронную сеть
1) Реализовать многослойную нейронную сеть с минимум 1 скрытым слоем (из 2+ нейронов) и выходным слоем с 1 нейроном для задачи XOR;
2) Реализовать многослойную нейронную сеть с минимум 1 скрытым слоем (из 25+ нейронов) и одним выходным слоем из 10 нейронов, для распознавания рукописных цифр MNIST (каждый нейрон выходного слоя распознает только свою цифру, то есть один нейрон распознает только 0, второй 1, третий 2 и т.д.);
3) Реализовать многослойную нейронную сеть на Keras (Документация: https://keras.io/, Пример с рукописными цифрами MNIST: https://keras.io/examples/vision/mnist_convnet/);
4) Нарисовать свою картинку цифры в Paint и подать в программу для распознавания (в свою нейронную сеть и в нейронку на Keras)

In [5]:
import torch
import torch.nn as nn
import numpy as np

### Часть 1
Задать случайные весовые параметры и обучить нейронную сеть для задачи XOR. Можно использовать пороговую функцию активации или sigmoid (можно другие).
Проверить учится ли многослойная нейронная сеть на прошлых задачах AND и OR. Многослойная нейронная сеть должна учиться для всех 3-х задач: AND, OR и XOR.
Все нейроны в сети должны учиться одновременно (Нельзя обучать один нейрон скрытого слоя на AND, второй на OR и склеивать в XOR. Все нейроны сети должны учиться одновременно (чисто на основе входных и выходных данных))

In [20]:
criterion = nn.MSELoss()

def train(model, optimizer, criterion, x, y, n_epochs=2001):
    y = y.view(-1, 1) # преобразую строку в столбец
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            print(f"Loss: {loss}")

In [24]:
X = torch.tensor([[0,0], [0,1], [1,0], [1,1]], dtype=torch.float32)
y_and = torch.tensor([0,0,0,1], dtype=torch.float32)
y_or = torch.tensor([0,1,1,1], dtype=torch.float32)
y_xor = torch.tensor([0,1,1,0], dtype=torch.float32)

print("Задача AND:")
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
train(model, optimizer, criterion, X, y_and)

print("Задача OR:")
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
train(model, optimizer, criterion, X, y_or)

print("Задача XOR:")
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Sigmoid(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
train(model, optimizer, criterion, X, y_xor)

Задача AND:
Loss: 0.20252805948257446
Loss: 3.747965092770755e-05
Loss: 1.2084705304005183e-05
Loss: 5.918034275964601e-06
Loss: 3.4445920391590334e-06
Задача OR:
Loss: 0.26100069284439087
Loss: 5.200089071877301e-05
Loss: 1.6201718608499505e-05
Loss: 7.918837582110427e-06
Loss: 4.6239938455983065e-06
Задача XOR:
Loss: 0.2595856487751007
Loss: 0.0002279290638398379
Loss: 7.30798055883497e-05
Loss: 3.6165503843221813e-05
Loss: 2.127198786183726e-05


### Часть 2

Реализовать многослойную нейронную сеть с минимум 1 скрытым слоем (из 25+ нейронов) и одним выходным слоем из 10 нейронов, для распознавания рукописных цифр MNIST (каждый нейрон выходного слоя распознает только свою цифру, то есть один нейрон распознает только 0, второй 1, третий 2 и т.д.)

In [57]:
from tqdm import tqdm

class MNISTDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return {
            "sample": torch.tensor(self.x[index, :], dtype=torch.float32),
            "target": self.y[index]
        }

class MNISTNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(1*28*28, 64)
        self.fc2 = nn.Linear(64, 10)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.flatten(x)
        preds = self.fc1(x)
        preds = self.sigmoid(preds)
        preds = self.fc2(preds)
        preds = self.sigmoid(preds)
        return preds

model = MNISTNeuralNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [58]:
import torchvision

mnist_train = torchvision.datasets.MNIST("./mnist/", train=True, download=True, transform=torchvision.transforms.ToTensor())
mnist_val = torchvision.datasets.MNIST("./mnist/", train=False, download=True, transform=torchvision.transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(mnist_train, batch_size=4, shuffle=True, num_workers=1)
val_loader = torch.utils.data.DataLoader(mnist_val, batch_size=4, shuffle=False, num_workers=1)

In [59]:
def train_nn(model, optimizer, criterion, n_epochs=5):
    for epoch in range(n_epochs):
        for x_train, y_train in tqdm(train_loader):
            optimizer.zero_grad()
            preds = model(x_train)
            y_train_one_hot = torch.nn.functional.one_hot(y_train, num_classes=10).float()
            loss = criterion(preds, y_train_one_hot)
            loss.backward()
            optimizer.step()
        if epoch % 2 == 0:
            val_loss = []
            val_accuracy = []
            with torch.no_grad():
                for x_val, y_val in tqdm(val_loader):
                    preds = model(x_val)
                    y_val_one_hot = torch.nn.functional.one_hot(y_val, num_classes=10).float()
                    loss = criterion(preds, y_val_one_hot)
                    accuracy = (preds.argmax(dim=1) == y_val).float().mean()
                    val_loss.append(loss)
                    val_accuracy.append(accuracy)
            print(f"Epoch: {epoch}, loss: {np.mean(val_loss)}, accuracy: {np.mean(val_accuracy)}")

In [60]:
train_nn(model, optimizer, criterion)

  0%|          | 0/15000 [00:00<?, ?it/s]

100%|██████████| 2500/2500 [00:12<00:00, 204.38it/s]


Epoch: 0, loss: 0.011471668258309364, accuracy: 0.930400013923645


100%|██████████| 2500/2500 [00:11<00:00, 220.46it/s]


Epoch: 2, loss: 0.01003348734229803, accuracy: 0.9429000020027161


100%|██████████| 2500/2500 [00:11<00:00, 221.89it/s]

Epoch: 4, loss: 0.009950358420610428, accuracy: 0.947700023651123


In [70]:
from PIL import Image

img = Image.open("6.jpg").convert("L")
transform = torchvision.transforms.ToTensor()
tensor_img = transform(img).unsqueeze(0) # приводим к размерности батча: (1, 1, 28, 28)

model.eval()
with torch.no_grad():
    preds = model(tensor_img)
    predicted_class = preds.argmax(dim=1).item()

    print(f"Сырые выходы модели: {preds}")
    print(f"Предсказанный класс: {predicted_class}")

img

Сырые выходы модели: tensor([[9.5340e-05, 1.5862e-09, 2.5254e-05, 6.9444e-09, 6.0075e-09, 1.0797e-01,
         2.1541e-01, 6.4957e-12, 5.5536e-03, 2.2930e-07]])
Предсказанный класс: 6


### Часть 3
Keras

In [71]:
import keras
from keras import layers
import tensorflow as tf

In [72]:
(x_train, y_train), (x_val, y_val) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0

y_train_one_hot = keras.utils.to_categorical(y_train, num_classes=10)
y_val_one_hot = keras.utils.to_categorical(y_val, num_classes=10)

In [73]:
model = keras.Sequential([
    layers.Flatten(input_shape=(28,28)),
    layers.Dense(64, activation="sigmoid"),
    layers.Dense(10, activation="sigmoid"),
])

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=0.01),
    loss = keras.losses.MeanSquaredError(),
    metrics = [keras.metrics.CategoricalAccuracy(name="accuracy")]
)

model.fit(
    x_train, y_train_one_hot,
    batch_size = 4,
    epochs = 5,
    validation_data = (x_val, y_val_one_hot),
    shuffle = True
)

c:\Users\Артем\Desktop\VUZ\МАГИСТРАТУРА\Совр.платф.прогр\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 44s 3ms/step - accuracy: 0.9131 - loss: 0.0146 - val_accuracy: 0.9363 - val_loss: 0.0111
Epoch 2/5
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - accuracy: 0.9382 - loss: 0.0109 - val_accuracy: 0.9329 - val_loss: 0.0116
Epoch 3/5
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 50s 3ms/step - accuracy: 0.9444 - loss: 0.0100 - val_accuracy: 0.9434 - val_loss: 0.0103
Epoch 4/5
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 51s 3ms/step - accuracy: 0.9463 - loss: 0.0096 - val_accuracy: 0.9433 - val_loss: 0.0101
Epoch 5/5
15000/15000 ━━━━━━━━━━━━━━━━━━━━ 46s 3ms/step - accuracy: 0.9486 - loss: 0.0093 - val_accuracy: 0.9505 - val_loss: 0.0091


In [79]:
img = Image.open("6.jpg").convert("L")
img_array = np.array(img).astype("float32") / 255.0
img_array = np.expand_dims(img_array, axis=0)

preds = model.predict(img_array)
predicted_class = np.argmax(preds, axis=1)[0]

print(f"Сырые выходы модели: {preds}")
print(f"Предсказанный класс: {predicted_class}")
img

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Сырые выходы модели: [[1.9850507e-03 1.0553630e-05 6.0455670e-04 1.3597653e-07 2.1769995e-09
  9.2885512e-01 1.0550016e-04 3.2969901e-12 2.2481254e-01 1.0304315e-08]]
Предсказанный класс: 5


### Часть 4
Заинференсить свой пример. Я выбрал цифру 6 с размером (28х28). Файл: 6.jpg

PyTorch: 12 ячейка

Keras: 17 ячейка

Несмотря на немного больший accuracy у модели, реализованной на Keras, она предсказала класс неверно. Модель, реализованная на PyTorch сделала верное предсказание.